In [7]:
import json
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
sphere_sample ={
    'hydrophobic_moment': [0.6, 1.00],
    'net_charge': [0.4, 0.6],
}

fiber_sample = {
    'has_beta_sheet_content': [0.1, 1],
    'net_charge': [0.4, 0.6],
}

samples = [sphere_sample, fiber_sample]

In [9]:
# read txt file with list of peptides
with open('generated_random_fibers_init.txt', 'r') as f:
    peptides_f = [line.strip() for line in f.readlines()]

len(set(peptides_f))

4800

In [10]:
# read txt file with list of peptides
with open('generated_random_spheres_init.txt', 'r') as f:
    peptides_s = [line.strip() for line in f.readlines()]

with open('generated_random_spheres_init_extra.txt', 'r') as f:
    peptides_s_extra = [line.strip() for line in f.readlines()]

peptides_s = peptides_s + peptides_s_extra
# remove peptides with length < 5
peptides_s = [pep for pep in peptides_s if len(pep) >= 5]
len(set(peptides_s))

4798

In [11]:
min_max = {
    "has_beta_sheet_content": (0.0, 1.0),
    "hydrophobic_moment": (0.000000, 1.998000),
    "net_charge": (-6.000000, 6.000000),
    "ap":(0.959986, 2.897030)
}

for sample in samples:
    for key in sample:
        if key in ['length','sequence']:
            continue
        sample[key][0] = sample[key][0] * (min_max[key][1] - min_max[key][0]) + min_max[key][0]
        sample[key][1] = sample[key][1] * (min_max[key][1] - min_max[key][0]) + min_max[key][0]

samples

[{'hydrophobic_moment': [1.1987999999999999, 1.998],
  'net_charge': [-1.1999999999999993, 1.1999999999999993]},
 {'has_beta_sheet_content': [0.1, 1.0],
  'net_charge': [-1.1999999999999993, 1.1999999999999993]}]

In [12]:
fiber_samples = pd.read_csv('gen_peptides/fibers_metrics_unconditional.csv')
fiber_samples['has_beta_sheet_content'] = (fiber_samples['beta_sheet_fraction'] > 0.1).astype(int)
fiber_samples.drop(columns=['peptide_id'], inplace=True)
fiber_samples.drop_duplicates(subset=['sequence'], inplace=True)
fiber_samples

,sequence,beta_sheet_fraction,hydrophobic_moment,net_charge,has_beta_sheet_content
0,INYPFYVG,0.0,0.498750,0,0
1,KDWVDKI,0.0,0.218571,0,0
2,KCCWMCDC,0.0,0.026250,0,0
3,AMWAGYFA,0.0,0.655000,0,0
4,EWIHCLPN,0.0,0.217500,-1,0
...,...,...,...,...,...
2038,LVPCTIQVY,0.0,0.485556,0,0
2039,QLSVHMF,0.0,0.362857,0,0
2040,VPPGFYV,0.0,0.618571,0,0
2041,TWRLMFS,0.0,0.134286,1,0


In [12]:
valid_fibers = fiber_samples.copy()

for key in fiber_sample:
    valid_fibers = valid_fibers[(valid_fibers[key] >= fiber_sample[key][0]) & ((valid_fibers[key] <= fiber_sample[key][1]))]

valid_fibers

,sequence,beta_sheet_fraction,hydrophobic_moment,net_charge,has_beta_sheet_content
118,FMVYDGKHWY,0.5,0.192,0,1
409,VYLSLHAPIY,0.5,0.526,0,1
650,KVEFAGTCAW,0.5,0.280,0,1
825,FCWTDHTKYN,0.5,0.113,0,1
1376,PCQIRNVFFF,0.5,0.228,1,1
1742,YFVWGDMKYN,0.5,0.154,0,1
1765,VFVFGSDTYT,0.5,0.410,-1,1


In [14]:
peptide_propensities_fibers = pd.read_csv('./filtered_ap_peptides_random_fiber.txt', names=['sequence', 'sequence_3', 'ap', 'assembly'])
peptide_propensities_fibers['ap'] = min_max['ap'][0] + (peptide_propensities_fibers['ap'] * (min_max['ap'][1] - min_max['ap'][0]))
peptide_propensities_fibers

,sequence,sequence_3,ap,assembly
0,YFDHASPQI,Tyr-Phe-Asp-His-Ala-Ser-Pro-Gln-Ile,1.819629,3.013318
1,HWRFMDIS,His-Trp-Arg-Phe-Met-Asp-Ile-Ser,1.927976,5.934383
2,YWVTIISIN,Tyr-Trp-Val-Thr-Ile-Ile-Ser-Ile-Asn,1.926437,7.408538
3,KTVSPFDL,Lys-Thr-Val-Ser-Pro-Phe-Asp-Leu,1.915634,4.207919
4,ACGPVWH,Ala-Cys-Gly-Pro-Val-Trp-His,1.946850,5.985378
...,...,...,...,...
2038,WMHPKGIG,Trp-Met-His-Pro-Lys-Gly-Ile-Gly,1.880830,3.928980
2039,DCLWFLSF,Asp-Cys-Leu-Trp-Phe-Leu-Ser-Phe,1.948079,10.866736
2040,LYVYYFGS,Leu-Tyr-Val-Tyr-Tyr-Phe-Gly-Ser,2.041111,12.297546
2041,MWHIVNHTNI,Met-Trp-His-Ile-Val-Asn-His-Thr-Asn-Ile,1.873249,5.147299


In [15]:
merged_valid_fibers = valid_fibers.merge(peptide_propensities_fibers, on='sequence', how='inner')
merged_valid_fibers.sort_values(by='ap', ascending=False)[['sequence', 'sequence_3', 'ap']][:15]

,sequence,sequence_3,ap
1,VYLSLHAPIY,Val-Tyr-Leu-Ser-Leu-His-Ala-Pro-Ile-Tyr,1.984795
0,FMVYDGKHWY,Phe-Met-Val-Tyr-Asp-Gly-Lys-His-Trp-Tyr,1.962766
5,YFVWGDMKYN,Tyr-Phe-Val-Trp-Gly-Asp-Met-Lys-Tyr-Asn,1.901199
4,PCQIRNVFFF,Pro-Cys-Gln-Ile-Arg-Asn-Val-Phe-Phe-Phe,1.877359
6,VFVFGSDTYT,Val-Phe-Val-Phe-Gly-Ser-Asp-Thr-Tyr-Thr,1.857780
3,FCWTDHTKYN,Phe-Cys-Trp-Thr-Asp-His-Thr-Lys-Tyr-Asn,1.853992
2,KVEFAGTCAW,Lys-Val-Glu-Phe-Ala-Gly-Thr-Cys-Ala-Trp,1.814311


In [16]:
sphere_samples = pd.read_csv('gen_peptides/spheres_metrics_unconditional.csv')
sphere_samples.drop(columns=['peptide_id'], inplace=True)
sphere_samples.drop_duplicates(subset=['sequence'], inplace=True)
sphere_samples

,sequence,beta_sheet_fraction,hydrophobic_moment,net_charge
0,KGSWG,0.0,0.018000,1
1,HAWIA,0.0,0.606000,0
2,PMAWE,0.0,0.290000,-1
3,PWGAWL,0.0,0.650000,0
4,CEWCF,0.0,0.368000,-1
...,...,...,...,...
2561,WIDLW,0.0,0.632000,-1
2562,KGSHI,0.0,0.044000,1
2563,LTFKW,0.0,0.302000,1
2564,IDPYNKY,0.0,0.165714,0


In [17]:
valid_spheres = sphere_samples.copy()

for key in sphere_sample:
    valid_spheres = valid_spheres[(valid_spheres[key] >= sphere_sample[key][0]) & ((valid_spheres[key] <= sphere_sample[key][1]))]

valid_spheres

,sequence,beta_sheet_fraction,hydrophobic_moment,net_charge
1395,FFIFFL,0.0,1.200,0
1972,IIFVI,0.0,1.282,0
1974,FFVII,0.0,1.244,0


In [18]:
peptide_propensities_spheres = pd.read_csv('./filtered_ap_peptides_random_spheres.txt', names=['sequence', 'sequence_3', 'ap', 'assembly'])
peptide_propensities_spheres_extra = pd.read_csv('./filtered_ap_peptides_random_spheres_extra.txt', names=['sequence', 'sequence_3', 'ap', 'assembly'])
peptide_propensities_spheres = pd.concat([peptide_propensities_spheres, peptide_propensities_spheres_extra], ignore_index=True)
peptide_propensities_spheres['ap'] = min_max['ap'][0] + (peptide_propensities_spheres['ap'] * (min_max['ap'][1] - min_max['ap'][0]))
peptide_propensities_spheres

,sequence,sequence_3,ap,assembly
0,SCTQW,Ser-Cys-Thr-Gln-Trp,1.958286,5.101487
1,VCHLCQS,Val-Cys-His-Leu-Cys-Gln-Ser,1.860669,4.525917
2,LRGLNW,Leu-Arg-Gly-Leu-Asn-Trp,1.813956,1.740523
3,ICYQYT,Ile-Cys-Tyr-Gln-Tyr-Thr,2.005406,8.069376
4,KWLQYMT,Lys-Trp-Leu-Gln-Tyr-Met-Thr,1.930065,5.851066
...,...,...,...,...
3160,NLIHCP,Asn-Leu-Ile-His-Cys-Pro,1.893499,4.241797
3161,WYWRLA,Trp-Tyr-Trp-Arg-Leu-Ala,2.011099,8.723896
3162,PVWAG,Pro-Val-Trp-Ala-Gly,2.045524,6.758625
3163,GYSYF,Gly-Tyr-Ser-Tyr-Phe,2.094711,9.504639


In [19]:
merged_valid_spheres = valid_spheres.merge(peptide_propensities_spheres, on='sequence', how='inner')
merged_valid_spheres.sort_values(by='ap', ascending=False)[['sequence', 'sequence_3', 'ap']][:15]

,sequence,sequence_3,ap
2,FFVII,Phe-Phe-Val-Ile-Ile,2.287872
0,FFIFFL,Phe-Phe-Ile-Phe-Phe-Leu,2.211490
1,IIFVI,Ile-Ile-Phe-Val-Ile,2.194745


In [20]:
merged_valid_spheres.to_csv('valid_spheres_unconditional.csv', index=False)
merged_valid_fibers.to_csv('valid_fibers_unconditional.csv', index=False)